In [ ]:
!pip install datasets
from datasets import load_dataset
import pandas as pd

raw_dataset = load_dataset("wangrongsheng/ag_news")

df= pd.DataFrame(raw_dataset['train'])
print(df.head())

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2


In [11]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

train_data = raw_dataset['train']
test_data = raw_dataset['test']

train_texts = [clean_text(x["text"]) for x in train_data]
test_texts = [clean_text(x["text"]) for x in test_data]

train_labels = [x["label"] for x in train_data]
test_labels = [x["label"] for x in test_data]

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")

tokenizer.fit_on_texts(train_texts)

X_train = tokenizer.texts_to_sequences(train_texts)
X_test = tokenizer.texts_to_sequences(test_texts)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer


tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['text'])
sequences = tokenizer.texts_to_sequences(df['text'])
vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary Size:", vocab_size)
print("First Sequence:", sequences[0])

Vocabulary Size: 70345
First Sequence: [442, 441, 1681, 14528, 108, 64, 1, 850, 21, 21, 753, 8196, 442, 6640, 10231, 2927, 4, 5810, 25989, 40, 4049, 797, 332]


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_length = 50

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print("Shape of X:", X.shape)
print(X[0])

Shape of X: (120000, 50)
[  442   441  1681 14528   108    64     1   850    21    21   753  8196
   442  6640 10231  2927     4  5810 25989    40  4049   797   332     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0]


In [ ]:
from tensorflow.keras.utils import to_categorical
y = to_categorical(df['label'])

print("Shape of y:", y.shape)
print(y[:5])

Shape of y: (120000, 4)
[[0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(96000, 50)
(24000, 50)


In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    SimpleRNN(64),
    Dense(4, activation='softmax')
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [19]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 166s 136ms/step - accuracy: 0.8194 - loss: 0.5289 - val_accuracy: 0.8797 - val_loss: 0.4007
Epoch 2/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 163s 136ms/step - accuracy: 0.9063 - loss: 0.3187 - val_accuracy: 0.8804 - val_loss: 0.4086
Epoch 3/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 202s 136ms/step - accuracy: 0.9282 - loss: 0.2547 - val_accuracy: 0.8734 - val_loss: 0.3897
Epoch 4/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 162s 135ms/step - accuracy: 0.9210 - loss: 0.2884 - val_accuracy: 0.8822 - val_loss: 0.4507
Epoch 5/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 162s 135ms/step - accuracy: 0.6578 - loss: 0.9044 - val_accuracy: 0.7006 - val_loss: 0.8546


In [20]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.7059 - loss: 0.8507
Test Loss: 0.8506608605384827
Test Accuracy: 0.7059166431427002
